# Cross-Validation: Comparing ResNet18 vs MobileNetV2

**Standalone notebook** (clones the repo and re-downloads data fresh).
**GPU needed this time** - `Runtime > Change runtime type > GPU` (T4 is fine).

**What this notebook does:** rebuilds the exact same leakage-safe splits
from notebook 03 (deterministic - same code, same seeds), then runs grouped
5-fold cross-validation for BOTH models under the identical training
protocol, to compare them before committing to final training. See
`docs/training.md` for the full reasoning behind every choice here
(staged transfer learning, hyperparameters, why CV is split into its own
notebook rather than continuing straight into final training).

**Compute flag:** this runs 10 total training runs (5 folds x 2 models),
each up to 30 epochs (usually fewer - early stopping). This is the
heaviest step in the project so far. If any fold is taking dramatically
longer than the others, or the whole notebook is clearly going to take
hours rather than well under one, stop and let Claude know rather than
letting it run indefinitely - that would mean something is wrong, not
that it's just slow.

**Nothing here becomes the final model.** This notebook only compares the
two architectures on the development set; the actual final models get
trained separately, after reviewing these results together.


## 1. Get the project code and dependencies

In [ ]:
import os

REPO_URL = "https://github.com/aural0i/Sickle-cell-detection"
BRANCH = "claude/sickle-cell-cnn-research-g6fipc"
if not os.path.isdir("/content/Sickle-cell-detection"):
    !git clone --branch {BRANCH} {REPO_URL} /content/Sickle-cell-detection
else:
    !git -C /content/Sickle-cell-detection pull
%cd /content/Sickle-cell-detection

import sys
sys.path.insert(0, "/content/Sickle-cell-detection")


In [ ]:
# Same lesson as notebooks 01/03: Colab already has everything except these two.
!pip install -q kaggle imagehash
print("Dependencies installed.")

import torch
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: no GPU detected - go to Runtime > Change runtime type > GPU before continuing.")
else:
    print("GPU:", torch.cuda.get_device_name(0))


## 2. Download the primary dataset and rebuild the splits

Identical process to notebook 03 (same code, same seeds -> same result,
deterministically). Paste your Kaggle API token (`KGAT_...`) when prompted.


In [ ]:
from getpass import getpass
import kaggle

os.environ["KAGGLE_API_TOKEN"] = getpass("Paste your Kaggle API token (KGAT_...): ")
kaggle.api.authenticate()

os.makedirs("data/train_source", exist_ok=True)
kaggle.api.dataset_download_files(
    "florencetushabe/sickle-cell-disease-dataset",
    path="data/train_source",
    unzip=True,
)
print("Download complete.")


In [ ]:
from src.data import (
    build_primary_manifest, find_duplicate_groups,
    make_held_out_test_split, make_cv_folds,
)

manifest = build_primary_manifest("data/train_source")
group_ids, exact_hashes, phashes = find_duplicate_groups(manifest["path"])
manifest["group_id"] = manifest["path"].map(group_ids)

dev_df, test_df = make_held_out_test_split(manifest, n_splits=5, seed=42)
dev_df = make_cv_folds(dev_df, n_splits=5, seed=43)

print(f"Development set: {len(dev_df)} images (held-out test set: {len(test_df)}, untouched)")
print(dev_df.groupby('cv_fold')['label_name'].value_counts().unstack())


## 3. Run grouped 5-fold CV for both models

Each fold: train on the other 4 folds, validate/early-stop on the held-out
fold. Same protocol for both models (see `docs/training.md`). Class weights
are recomputed per fold from that fold's own training portion only.


In [ ]:
import torch
from src.data import SickleCellDataset, compute_class_weights
from src.training import build_model, train_with_early_stopping, set_seed, save_history

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

TRAINING_KWARGS = dict(
    stages=(1, 2),
    epochs_per_stage=(15, 15),
    lr_per_stage=(1e-3, 1e-4),
    batch_size=32,
    patience=5,
    seed=42,
)

def run_cv_for_model(model_name, dev_df, n_folds=5):
    fold_results = []
    for fold in range(n_folds):
        print(f"\n=== {model_name} - fold {fold} ===")
        set_seed(TRAINING_KWARGS["seed"])

        train_split = dev_df[dev_df["cv_fold"] != fold].reset_index(drop=True)
        val_split = dev_df[dev_df["cv_fold"] == fold].reset_index(drop=True)

        train_ds = SickleCellDataset(train_split, train=True)
        val_ds = SickleCellDataset(val_split, train=False)
        weights = compute_class_weights(train_split["label"])

        model = build_model(model_name, pretrained=True)
        best_state, history = train_with_early_stopping(
            model, model_name, train_ds, val_ds, weights, device, **TRAINING_KWARGS
        )

        best_epoch = min(history, key=lambda h: h["val_loss"])
        print(f"Best epoch: {best_epoch['epoch']} (stage {best_epoch['stage']}) - "
              f"val_loss={best_epoch['val_loss']:.4f}, val_acc={best_epoch['val_acc']:.4f}")

        save_history(history, f"results/cv/{model_name}_fold{fold}_history.json")
        fold_results.append({
            "fold": fold,
            "val_loss": best_epoch["val_loss"],
            "val_acc": best_epoch["val_acc"],
            "n_epochs_run": len(history),
        })

        del model
        torch.cuda.empty_cache()

    return fold_results


In [ ]:
resnet_results = run_cv_for_model("resnet18", dev_df)


In [ ]:
mobilenet_results = run_cv_for_model("mobilenet_v2", dev_df)


## 4. Compare CV results

In [ ]:
import pandas as pd
import numpy as np

def summarize(results, name):
    df = pd.DataFrame(results)
    print(f"--- {name} ---")
    print(df)
    print(f"Mean val_loss: {df['val_loss'].mean():.4f}  (std: {df['val_loss'].std():.4f})")
    print(f"Mean val_acc:  {df['val_acc'].mean():.4f}  (std: {df['val_acc'].std():.4f})")
    print()
    return df

resnet_df = summarize(resnet_results, "ResNet18")
mobilenet_df = summarize(mobilenet_results, "MobileNetV2")

resnet_df.to_csv("results/cv/resnet18_cv_summary.csv", index=False)
mobilenet_df.to_csv("results/cv/mobilenet_v2_cv_summary.csv", index=False)
print("Saved per-fold summaries to results/cv/")


## 5. Plot training curves (one representative fold per model)

Shows train vs. validation loss per epoch, to check for overfitting
(train loss much lower than val loss and diverging), underfitting (both
stay high), or stable learning (both decrease and roughly track together).
The vertical line marks the stage 1 -> stage 2 transition.


In [ ]:
import json
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model_name in zip(axes, ["resnet18", "mobilenet_v2"]):
    with open(f"results/cv/{model_name}_fold0_history.json") as f:
        history = json.load(f)
    epochs = [h["epoch"] for h in history]
    train_loss = [h["train_loss"] for h in history]
    val_loss = [h["val_loss"] for h in history]
    stage_change = next((h["epoch"] for h in history if h["stage"] == 2), None)

    ax.plot(epochs, train_loss, label="train loss")
    ax.plot(epochs, val_loss, label="val loss")
    if stage_change:
        ax.axvline(stage_change - 0.5, color="gray", linestyle="--", label="stage 1 -> 2")
    ax.set_title(f"{model_name} (fold 0)")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend()

plt.tight_layout()
plt.savefig("results/cv/training_curves_fold0.png", dpi=100)
plt.show()


## 6. What to do with this output

Copy back to Claude:
- The per-fold results and mean/std for both models (Section 4)
- Whether the training curves (Section 5) look like stable learning,
  overfitting, or underfitting
- How long this actually took to run, roughly

**This is the checkpoint before final training.** Once we've looked at
these results together, Claude will write the next notebook to train both
models on the full development set and save the checkpoints - nothing has
been finalized yet.
